In [0]:
CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.gold_kpi_tarefas_empresa AS
SELECT
    e.id_dim_empresa,
    e.nome_empresa,
    e.setor_empresa,
    SUM(f.qtd_tarefas) AS total_tarefas,
    SUM(f.qtd_concluidas) AS total_concluidas,
    SUM(f.qtd_em_andamento) AS total_em_andamento,
    SUM(f.qtd_bloqueadas) AS total_bloqueadas,
    SUM(f.qtd_atrasadas) AS total_atrasadas,

    ROUND(100.0 * SUM(f.qtd_concluidas)/ NULLIF(SUM(f.qtd_tarefas), 0),2) AS taxa_conclusao_pct,

    ROUND(100.0 * SUM(f.qtd_atrasadas)/ NULLIF(SUM(f.qtd_tarefas), 0),2) AS taxa_atraso_pct,

    ROUND(100.0 * SUM(f.qtd_bloqueadas)/ NULLIF(SUM(f.qtd_tarefas), 0),2) AS taxa_bloqueio_pct

FROM workspace.silver.silver_fato_tarefas f
JOIN workspace.silver.silver_empresa e ON f.id_dim_empresa = e.id_dim_empresa
GROUP BY
    e.id_dim_empresa,
    e.nome_empresa,
    e.setor_empresa;


CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.gold_evolucao_tarefas AS
SELECT
    f.id_dim_data,
    f.id_dim_empresa,
    e.nome_empresa,
    SUM(f.qtd_tarefas) AS total_tarefas,
    SUM(f.qtd_concluidas) AS total_concluidas,
    SUM(f.qtd_em_andamento) AS total_em_andamento,
    SUM(f.qtd_bloqueadas) AS total_bloqueadas,
    SUM(f.qtd_atrasadas) AS total_atrasadas,

    ROUND(100.0 * SUM(f.qtd_concluidas)/ NULLIF(SUM(f.qtd_tarefas), 0),2) AS taxa_conclusao_pct

FROM workspace.silver.silver_fato_tarefas f
JOIN workspace.silver.silver_empresa e ON f.id_dim_empresa = e.id_dim_empresa
WHERE f.id_dim_data IS NOT NULL
GROUP BY
    f.id_dim_data,
    f.id_dim_empresa,
    e.nome_empresa;


CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.gold_kpi_ciclos AS
SELECT
    id_ciclo,
    id_empresa,
    nome_empresa,
    titulo AS titulo_ciclo,
    status AS status_ciclo,
    data_inicio,
    data_estimada_fim,
    data_fim_real,
    COALESCE(qtd_problemas, 0) AS qtd_problemas,
    COALESCE(qtd_causas, 0) AS qtd_causas,
    COALESCE(qtd_planos, 0) AS qtd_planos,
    COALESCE(planos_concluidos, 0) AS planos_concluidos,
    COALESCE(qtd_tarefas, 0) AS qtd_tarefas,
    COALESCE(tarefas_concluidas, 0) AS tarefas_concluidas,
    COALESCE(tarefas_atrasadas, 0) AS tarefas_atrasadas,
    COALESCE(qtd_metas, 0) AS qtd_metas,
    COALESCE(metas_atingidas, 0) AS metas_atingidas,

    ROUND(100.0 * COALESCE(tarefas_concluidas, 0)/ NULLIF(qtd_tarefas, 0),2) AS taxa_conclusao_tarefas_pct,

    ROUND(100.0 * COALESCE(planos_concluidos, 0)/ NULLIF(qtd_planos, 0),2) AS taxa_conclusao_planos_pct,

    ROUND(100.0 * COALESCE(metas_atingidas, 0)/ NULLIF(qtd_metas, 0),2) AS taxa_metas_atingidas_pct

FROM workspace.silver.silver_resumo_ciclo;


CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.gold_kpi_bloqueios AS
SELECT
    id_empresa,
    nome_empresa,
    COUNT(DISTINCT id_tarefa) AS total_tarefas_bloqueadas
FROM workspace.silver.silver_tarefas_bloqueadas
WHERE id_tarefa IS NOT NULL
GROUP BY nome_empresa, id_empresa;


CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.gold_kpi_treinamentos AS
SELECT
    e.id_dim_empresa,
    e.nome_empresa,
    e.setor_empresa,
    SUM(COALESCE(f.qtd_inscricoes, 0)) AS total_inscricoes,
    SUM(COALESCE(f.qtd_realizados, 0)) AS total_realizados,

    ROUND(100.0 * SUM(COALESCE(f.qtd_realizados, 0))/ NULLIF(SUM(COALESCE(f.qtd_inscricoes, 0)), 0),2) AS taxa_realizacao_pct

FROM workspace.silver.silver_fato_treinamentos f
JOIN workspace.silver.silver_empresa e ON f.id_dim_empresa = e.id_dim_empresa
GROUP BY
    e.id_dim_empresa,
    e.nome_empresa,
    e.setor_empresa;


CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.gold_dau_empresa AS
SELECT
    data_atv,
    id_empresa,
    nome_empresa,
    dau
FROM workspace.silver.silver_dau_empresa
WHERE data_atv IS NOT NULL
  AND id_empresa IS NOT NULL;

CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.gold_kpi_desempenho_por_estado
COMMENT "Indicadores consolidados de empresas e tarefas agrupados por Estado (UF)"
AS
SELECT
    en.uf,
    COUNT(DISTINCT f.id_dim_empresa) AS total_empresas,
    SUM(f.qtd_tarefas) AS total_tarefas,
    SUM(f.qtd_concluidas) AS total_concluidas,
    ROUND(100.0 * SUM(f.qtd_concluidas) / NULLIF(SUM(f.qtd_tarefas), 0), 2) AS taxa_conclusao_pct
FROM workspace.silver.silver_fato_tarefas f
JOIN workspace.silver.silver_endereco en ON f.id_dim_empresa = en.id_empresa
WHERE en.principal = TRUE
GROUP BY en.uf;
